# 4.1 Demo - Course Setup and Authentication
**Course:** Automated Deployment with Declarative Automation Bundles · **Lessons:** DevOps Review · CI/CD Review · Course Setup and Authentication · Deploying Databricks Projects

Bundles are deployed **from your machine or a CI runner** with the **Databricks CLI**. Everything in this course works in two modes:

| Mode | What runs |
|---|---|
| **Offline** (no workspace) | Reading and editing bundles, `adelab.bundles.preview()` (resolved config per target), schema validation with the CLI's own JSON schema, unit tests |
| **Workspace** (recommended: free [Databricks Free Edition](https://www.databricks.com/learn/free-edition)) | `databricks bundle validate / deploy / run / destroy`, exactly as in the course videos |

Cells that need a workspace detect it and print `SKIPPED` until you log in.

## A. Deployment options at a glance (lesson: Deploying Databricks Projects)
| Tool | Good for | Downsides |
|---|---|---|
| UI (click-ops) | exploration | not repeatable, no review, no history |
| REST API | anything, from any language | you write and maintain all the orchestration |
| Python SDK (`databricks-sdk`) | automation scripts, apps | imperative: you manage create vs. update vs. delete |
| CLI | scripting single objects | same as SDK |
| **Bundles** (CLI `bundle` commands) | **projects as code**: jobs, pipelines, dashboards, models, apps + their source files, per environment | YAML to learn |

Bundles are **declarative**. You describe the desired state and `deploy` works out the create, update, and delete calls (Terraform-like state lives in the workspace).

![Deployment options from UI to bundles, imperative to declarative](../images/4.1-deployment-options.svg)

**How to read it:** left to right, each option needs less hand-written orchestration. The four grey tools are imperative: your script decides
every create, update, and delete call. Bundles (blue) are declarative: the YAML in `bundles/` states the result and
`databricks bundle deploy` computes the calls. All of them use the same CLI you check in the next cell.

In [ ]:
from adelab import bundles

print("CLI:", bundles.cli_path())
bundles.cli("--version")

## B. Authenticate (one time)
Run these **in a terminal** (they open a browser). Never paste tokens into notebooks or files.

**1. User-to-machine OAuth (U2M)**, for your own interactive work:
```powershell
databricks auth login --host https://<your-workspace>.cloud.databricks.com --profile DEFAULT
```
This creates a profile in `~/.databrickscfg` (`%USERPROFILE%\.databrickscfg` on Windows) that holds the host and points to cached OAuth tokens. It does **not** store a password.

**2. Machine-to-machine OAuth (M2M)**, for CI/CD (lesson 4.7). A **service principal** with an OAuth secret, passed via environment variables by the CI system:
```
DATABRICKS_HOST, DATABRICKS_CLIENT_ID, DATABRICKS_CLIENT_SECRET
```
Personal access tokens (PATs) still work but are discouraged. Prefer OAuth.

![U2M OAuth for developers versus M2M OAuth with a service principal for CI](../images/4.1-authentication.svg)

**How to read it:** the top row is what you do once now: `databricks auth login` writes a profile to `~/.databrickscfg`, which the next
cell lists with `auth profiles`. The bottom row is what CI does in lesson 4.7: the three secrets (red, names only) become
environment variables and the CLI signs in as a service principal. The workspace sees a different identity per path.

In [ ]:
bundles.cli("auth", "profiles")

In [ ]:
status = bundles.workspace_status()
status

In [ ]:
if bundles.require_workspace(status):
    bundles.cli("current-user", "me")
    bundles.cli("catalogs", "list")

## C. DevOps and CI/CD in one picture
![DevOps loop from code through CI, stage, approval, prod and monitoring](../images/4.1-devops-cicd-loop.svg)

**How to read it:** follow the numbers. Steps 2-3 (green) are CI and must pass, or the red dashed path sends you back to fix the code.
Steps 4-7 are CD: the same commit goes to stage (amber, `bundle deploy -t stage` + a test run), then to prod (orange)
after approval. Monitoring feeds the next change. Lesson 4.7 builds exactly this for `bundles/03_cicd_dab`.

- **Continuous integration**: every change is built and tested automatically (unit tests run locally on Spark, as in `bundles/03_cicd_dab/tests`)
- **Continuous delivery / deployment**: every merged change is deployed automatically to stage, and prod after approval
- **Environments** (dev / stage / prod) = bundle **targets** with different variables, permissions, and identities

Next: **4.2 Deploying a Simple DAB**.